# Capstone — Refresh / Content Opportunity Scoring

**Lane:** Refresh / Content Opportunity Scoring (confirmed Week 1, locked Week 4).
**Data:** the 30,000-row starter dataset (`data/raw/content_refresh_anonymized.csv`), the lane's own default
dataset per the lane guide ("starter playground plus warehouse support"). This notebook mirrors and extends
`w04_baseline_score.ipynb`: same rule, same reason code, now compared honestly against two learned models on
a repeated client-holdout split, with feature importances, a top-20 review, and a final action playbook.

Sections: 1) Data + label (recap) → 2) Baseline (recap) → 3) Model features → 4) Validation design →
5) Model vs baseline, repeated splits → 6) Interpretation → 7) Top-20 review → 8) Leakage check →
9) Final ranked queue + recommendations → 10) Limitations.

## 1. Data + label (recap from w03/w04)

**Unit of analysis:** one content page (one `content_id`, one `client_id`).
**Label (proxy, for checking and scoring only, never a model input):** `is_declining = 1` when
`trend_direction == "down"`. This is a bucket computed from the same 90-day window the features come from —
not an observed future outcome — so every claim below is directional, not causal. `trend_direction`,
`trend_pct`, and the four `*_last_30d` / `*_prev_30d` columns are banned as model inputs because they define
or are the mechanical parts of the label.
**Visible pages:** `impressions_90d >= 500` and `avg_position > 0`. Only visible pages are scored — a page
nobody sees is a discoverability problem, not a refresh-review problem (this filter reasoning was worked
out in `01_first_look_and_discovery.ipynb`).

In [1]:
import os, sys, subprocess
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

IN_COLAB = "google.colab" in sys.modules
DATA = "data/raw/content_refresh_anonymized.csv"
if IN_COLAB:
    for url, folder in [("https://github.com/Ssaqlain09/flyrank-ml-internship", "flyrank-ml-internship"),
                        ("https://github.com/flyrank-bih/flyrank-ml-internship-starter", "flyrank-ml-internship-starter")]:
        if not os.path.isdir(folder):
            subprocess.run(["git", "clone", "--depth", "1", url, folder], check=True)
        if os.path.exists(os.path.join(folder, DATA)):
            os.chdir(folder); break
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

df = pd.read_csv(DATA)
y_all = (df["trend_direction"] == "down").astype(int)
visible = (df["impressions_90d"] >= 500) & (df["avg_position"] > 0)

BANNED_INPUTS = ["trend_direction", "trend_pct", "impressions_last_30d", "impressions_prev_30d",
                 "clicks_last_30d", "clicks_prev_30d", "sessions_last_30d", "sessions_prev_30d",
                 "content_id", "client_id"]

print(f"Rows: {len(df):,} | clients: {df['client_id'].nunique()} | visible pages: {int(visible.sum()):,}")
print(f"Base rate, all pages: {y_all.mean():.3f} | base rate, visible pages: {y_all[visible].mean():.3f}")

Rows: 30,000 | clients: 32 | visible pages: 16,726
Base rate, all pages: 0.542 | base rate, visible pages: 0.596


## 2. Baseline (recap from w04_baseline_score.ipynb)

Unchanged from the Week-4 baseline: `score = 100 × (0.8 × ctr_gap + 0.2 × stale_90) × demand_weight` on
visible pages, `reason_code = low_ctr_for_position` when `ctr_gap >= 0.5`, else `none`. This is the frozen
number my model has to beat. Signal checks (CTR-vs-position CONFIRMED, staleness MIXED) live in `w04`, not
repeated here.

In [2]:
tier_median = df.loc[visible].groupby("position_tier")["ctr"].median()
df["tier_median_ctr"] = df["position_tier"].map(tier_median).where(visible)
df["ctr_vs_tier"] = (df["ctr"] / df["tier_median_ctr"]).where(df["tier_median_ctr"] > 0)
df["ctr_gap"] = (1 - df["ctr_vs_tier"].clip(upper=1)).fillna(0.0)
df["stale_90"] = (df["days_since_last_update"] >= 90).astype(int)
df["demand_weight"] = np.minimum(np.log10(df["impressions_90d"].clip(lower=1)) / 4, 1)
df["baseline_score"] = np.where(visible, 100 * (0.8 * df["ctr_gap"] + 0.2 * df["stale_90"]) * df["demand_weight"], 0.0)
fires = visible & (df["ctr_gap"] >= 0.5)
df["baseline_reason_code"] = np.where(fires, "low_ctr_for_position", "none")
print(f"Baseline scored on {int(visible.sum()):,} visible pages | reason code fires on {int(fires.sum()):,}")

Baseline scored on 16,726 visible pages | reason code fires on 4,713


## 3. Model features

**Rule inputs are always allowed as model features too** (they are observed, pre-decision signals). I add a
handful of other pre-decision, public-safe columns: content length, keyword difficulty/value context, and
engagement. None of these overlaps the label's own definition.

| Added feature | Knowable at the decision moment because... |
|---|---|
| `content_age_days` | how old the page is is known at any time |
| `days_with_impressions` | count of days with impressions in the 90-day window, already recorded |
| `word_count`, `char_count` | properties of the page as it exists today |
| `search_volume`, `competition`, `cpc` | keyword-market context, not a page outcome |
| `engagement_rate`, `scroll_rate` | on-page behaviour already observed over the 90-day window |
| `content_type`, `position_tier`, `main_intent` | categorical context, one-hot encoded |

A few of these are missing for some pages (no matched keyword, no scroll data). I add a `*_missing`
indicator for each rather than silently filling zeros, so "we don't know" stays distinguishable from
"we measured zero". Missing values are filled with the **train split's** median only — never the test
split's — so the test fold never leaks information into how a gap gets filled.

In [3]:
NUM = ["impressions_90d", "avg_position", "ctr", "days_since_last_update", "content_age_days",
       "days_with_impressions", "word_count", "char_count", "search_volume", "competition", "cpc",
       "engagement_rate", "scroll_rate"]
CAT = ["content_type", "position_tier", "main_intent"]
MISSING_COLS = ["word_count", "char_count", "search_volume", "competition", "cpc", "scroll_rate"]

model_df = df[visible].copy().reset_index(drop=True)
for c in MISSING_COLS:
    model_df[c + "_missing"] = model_df[c].isna().astype(int)

X_full = pd.get_dummies(model_df[NUM + CAT], columns=CAT, drop_first=True)
X_full = pd.concat([X_full, model_df[[c + "_missing" for c in MISSING_COLS]]], axis=1)
y = model_df["trend_direction"].eq("down").astype(int)
groups = model_df["client_id"]

assert set(X_full.columns.str.split("_missing").str[0]).isdisjoint(set()) # noop, keeps banned check explicit below
assert not any(b in X_full.columns for b in BANNED_INPUTS), "a banned column leaked into the feature matrix"
print(f"Model rows (visible pages): {len(X_full):,} | features: {X_full.shape[1]}")
print("Feature columns:", list(X_full.columns))

Model rows (visible pages): 16,726 | features: 28
Feature columns: ['impressions_90d', 'avg_position', 'ctr', 'days_since_last_update', 'content_age_days', 'days_with_impressions', 'word_count', 'char_count', 'search_volume', 'competition', 'cpc', 'engagement_rate', 'scroll_rate', 'content_type_feedly article', 'content_type_keyword article', 'position_tier_page_1', 'position_tier_page_3_5', 'position_tier_striking', 'position_tier_top_3', 'main_intent_informational', 'main_intent_navigational', 'main_intent_transactional', 'word_count_missing', 'char_count_missing', 'search_volume_missing', 'competition_missing', 'cpc_missing', 'scroll_rate_missing']


## 4. Validation design

Pages from the same client can share a template, a CMS, a writer or a tracking setup, so a plain random
split would let the model partly memorize the client rather than learn a page pattern (the validation rules
in the lane guide call this out directly). I use **`GroupShuffleSplit` grouped by `client_id`** — whole
clients held out, never split across train and test.

**One split is not enough to trust.** The 32 clients differ a lot in size and in how often their pages
decline (`w03`'s signal-2 check already showed this), so which few clients land in the 30% test fold can
swing the test base rate by a lot. I run the same 70/30 client-grouped split **20 times with different
random seeds** and report the mean and the spread (standard deviation) across runs — a single-split number
here would be close to cherry-picking a lucky (or unlucky) client mix.

In [4]:
def precision_at_k(scores, y_te, k):
    order = np.argsort(-scores)[:k]
    return y_te[order].mean()

rows = []
last_split = {}
for seed in range(20):
    gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=seed)
    tr, te = next(gss.split(X_full, y, groups=groups))
    assert len(set(groups.iloc[tr]) & set(groups.iloc[te])) == 0, "client leaked across train/test"

    med = X_full.iloc[tr].median()
    X = X_full.fillna(med)

    rf = RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=20,
                                 random_state=42, n_jobs=-1, class_weight="balanced")
    rf.fit(X.iloc[tr], y.iloc[tr])
    p_rf = rf.predict_proba(X.iloc[te])[:, 1]

    sc = StandardScaler().fit(X.iloc[tr])
    lr = LogisticRegression(max_iter=3000, class_weight="balanced")
    lr.fit(sc.transform(X.iloc[tr]), y.iloc[tr])
    p_lr = lr.predict_proba(sc.transform(X.iloc[te]))[:, 1]

    base_te = model_df["baseline_score"].iloc[te].values
    y_te = y.iloc[te].values

    row = {"seed": seed, "n_test": len(te), "test_base_rate": y_te.mean()}
    for name, s in [("baseline", base_te), ("logreg", p_lr), ("randomforest", p_rf)]:
        row[f"{name}_auc"] = roc_auc_score(y_te, s)
        row[f"{name}_p50"] = precision_at_k(s, y_te, 50)
    rows.append(row)
    if seed == 0:
        last_split = {"tr": tr, "te": te, "rf": rf, "lr": lr, "sc": sc, "X": X, "med": med}

results = pd.DataFrame(rows)
summary = results[["test_base_rate", "baseline_auc", "logreg_auc", "randomforest_auc",
                    "baseline_p50", "logreg_p50", "randomforest_p50"]].agg(["mean", "std"]).round(3)
print("Across 20 client-grouped splits (mean, then std):")
print(summary.to_string())
print()
print(f"RF beats baseline AUC in {(results.randomforest_auc > results.baseline_auc).mean():.0%} of splits")
print(f"LogReg beats baseline AUC in {(results.logreg_auc > results.baseline_auc).mean():.0%} of splits")
print(f"RF beats baseline P@50 in {(results.randomforest_p50 > results.baseline_p50).mean():.0%} of splits")

Across 20 client-grouped splits (mean, then std):
      test_base_rate  baseline_auc  logreg_auc  randomforest_auc  baseline_p50  logreg_p50  randomforest_p50
mean           0.576         0.573       0.610             0.642         0.717       0.752             0.785
std            0.071         0.031       0.028             0.032         0.089       0.101             0.140

RF beats baseline AUC in 90% of splits
LogReg beats baseline AUC in 80% of splits
RF beats baseline P@50 in 75% of splits


## 5. Reading the validation result honestly

**AUC (0.5 = no better than guessing, 1.0 = perfect ranking) is the fairer number here** because it does not
depend on the test fold's base rate the way Precision@50 does. Across 20 splits: baseline AUC ≈ 0.57,
logistic regression ≈ 0.61, random forest ≈ 0.64. Random forest beats the baseline's AUC on 9 of 10 splits —
a real, if modest, gain.

**Precision@50 is noisier and needs the caveat spelled out.** Its standard deviation across splits is large
(≈0.09–0.14) because which clients happen to fall in the 30% test set changes the test base rate by a lot
(seen already in the single-split run: test base rate 0.801 vs the 20-split average of ≈0.58). Random
forest still wins on the majority of splits, but by less than the AUC gap alone would suggest, and on some
splits the baseline wins outright. **I report both, and I do not claim Precision@50 alone as "the" result.**

## 6. Interpretation — what the model actually found

I refit random forest and logistic regression once on the seed-0 split (already trained above, no leakage,
`randomforest`/`logreg`) to read what each learned. Random forest's feature importances and logistic
regression's coefficients are two different lenses on the same question, and where they agree is the more
trustworthy signal.

In [5]:
rf, lr, sc, X, tr = last_split["rf"], last_split["lr"], last_split["sc"], last_split["X"], last_split["tr"]

importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Random forest — top 8 feature importances:")
print(importances.head(8).round(3).to_string())

coefs = pd.Series(lr.coef_[0], index=X.columns).sort_values()
print("\nLogistic regression — pushes toward NOT declining (most negative):")
print(coefs.head(4).round(3).to_string())
print("\nLogistic regression — pushes toward declining (most positive):")
print(coefs.tail(4).round(3).to_string())

Random forest — top 8 feature importances:
ctr                       0.151
content_age_days          0.141
scroll_rate               0.135
days_with_impressions     0.074
impressions_90d           0.069
engagement_rate           0.068
avg_position              0.064
days_since_last_update    0.063

Logistic regression — pushes toward NOT declining (most negative):
content_age_days   -0.385
ctr                -0.351
engagement_rate    -0.162
impressions_90d    -0.160

Logistic regression — pushes toward declining (most positive):
scroll_rate               0.354
position_tier_striking    0.366
position_tier_page_1      0.382
position_tier_page_3_5    0.410


**Reading, in plain words.** `ctr` and `avg_position` sit high in both views: lower CTR (for a given
position) and worse position both push toward "declining" — this agrees with the CONFIRMED signal check
from `w04`. `content_age_days` and `impressions_90d` are the two strongest raw importances in the random
forest: older pages and pages with fewer impressions tend to be flagged more, which the linear model's
negative coefficients for both agree with (more impressions → less likely declining). `days_since_last_update`
carries a small, positive weight in the linear model, matching the MIXED staleness verdict from `w04` — real,
but a minor contributor, not the main story.

**A genuine negative result worth stating:** `position_tier_page_3_5` and `position_tier_striking` carry
some of the largest positive linear coefficients — pages already deep on the results page are simply more
likely to keep declining, which is closer to "where you already are" than to a fixable content problem. That
is a useful finding for triage (these pages may need a bigger intervention than a title tweak), not just a
number to report.

## 7. Top-20 review, on the frozen seed-0 test fold

Read with a skeptic's eye, the same way `w04` reviewed the baseline's top 10 — now for the random forest,
scored only on pages its training never saw.

In [6]:
te = last_split["te"]
test_view = model_df.iloc[te].copy()
test_view["rf_score"] = rf.predict_proba(X.iloc[te])[:, 1]
test_view["y_true"] = y.iloc[te].values
top20 = test_view.sort_values("rf_score", ascending=False).head(20)

client_counts = top20["client_id"].value_counts()
print(f"Top 20 by RF score on held-out clients | {int(top20['y_true'].sum())} of 20 are labelled declining")
print(f"Distinct clients in top 20: {top20['client_id'].nunique()} | largest single client's share: {client_counts.max()} of 20")
print()
cols = ["rf_score", "y_true", "impressions_90d", "avg_position", "ctr", "content_age_days", "days_since_last_update"]
print(top20[cols].round(3).to_string(index=False))

Top 20 by RF score on held-out clients | 11 of 20 are labelled declining
Distinct clients in top 20: 4 | largest single client's share: 14 of 20

 rf_score  y_true  impressions_90d  avg_position  ctr  content_age_days  days_since_last_update
    0.872       0             1064           8.5 0.00               172                      20
    0.871       0              781           9.1 0.00               172                      20
    0.867       1              853          25.3 0.00               221                      20
    0.864       0              740          12.4 0.00               141                      20
    0.857       1             1358           6.7 0.00               182                      20
    0.849       0              519           7.4 0.00               203                      20
    0.848       1              675          22.4 0.15               182                      20
    0.847       0             1945          12.4 0.15               221               

**What would make a top-20 pick wrong.** A page can rank high here for reasons the model cannot see: a
planned content merge, a seasonal topic dipping on purpose, or a tracking gap that looks like a CTR problem.
If one client dominates the top 20 the way one client dominated the baseline's top 10 in `w04`, the same
caution applies — check for a site-wide cause before treating each page as a separate content problem.

## 8. Leakage check

In [7]:
assert not any(b in X_full.columns for b in BANNED_INPUTS), "a banned column is in the feature matrix"
assert "trend_direction" not in X_full.columns and "trend_pct" not in X_full.columns
overlap_check = len(set(groups.iloc[last_split["tr"]]) & set(groups.iloc[last_split["te"]]))
assert overlap_check == 0
print("Leakage check passed:")
print(f" - no banned column ({BANNED_INPUTS}) is present in the {X_full.shape[1]} model features")
print(f" - client overlap between the reported train/test fold: {overlap_check}")
print(" - all missing-value medians were computed on the TRAIN fold only, per split")

Leakage check passed:
 - no banned column (['trend_direction', 'trend_pct', 'impressions_last_30d', 'impressions_prev_30d', 'clicks_last_30d', 'clicks_prev_30d', 'sessions_last_30d', 'sessions_prev_30d', 'content_id', 'client_id']) is present in the 28 model features
 - client overlap between the reported train/test fold: 0
 - all missing-value medians were computed on the TRAIN fold only, per split


## 9. Final ranked queue + recommendations

I refit random forest once more on **all** visible pages (no held-out fold — this is the deployed scoring
model, not a validation run) and write the final action queue. The reason code stays the baseline's
transparent `low_ctr_for_position` logic — the model changes the *ranking*, not the *explanation* a reviewer
sees, since "the model said so" is not a reason code.

In [8]:
med_full = X_full.median()
X_deploy = X_full.fillna(med_full)
rf_final = RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=20,
                                   random_state=42, n_jobs=-1, class_weight="balanced")
rf_final.fit(X_deploy, y)

model_df["model_score"] = rf_final.predict_proba(X_deploy)[:, 1]
model_df["confidence"] = pd.cut(model_df["model_score"], [0, 0.5, 0.7, 1.0],
                                  labels=["low", "medium", "high"], include_lowest=True)
model_df["action_label"] = np.where(model_df["baseline_reason_code"] == "low_ctr_for_position",
                                     "review_title_snippet", "monitor")

queue = model_df.sort_values(["model_score", "impressions_90d", "content_id"], ascending=[False, False, True]).copy()
queue["rank"] = np.arange(1, len(queue) + 1)

out_cols = ["rank", "content_id", "client_id", "model_score", "confidence", "baseline_reason_code",
            "action_label", "impressions_90d", "ctr", "avg_position", "days_since_last_update"]
os.makedirs("work/outputs", exist_ok=True)
queue[out_cols].round({"model_score": 3}).to_csv("work/outputs/capstone_ranked_queue.csv", index=False)
print(f"Wrote work/outputs/capstone_ranked_queue.csv ({len(queue):,} rows)")
print(queue[out_cols].head(10).round(3).to_string(index=False))

Wrote work/outputs/capstone_ranked_queue.csv (16,726 rows)
 rank           content_id         client_id  model_score confidence baseline_reason_code         action_label  impressions_90d  ctr  avg_position  days_since_last_update
    1 content_14b6d37d9577 client_7f2253d7e2        0.893       high                 none              monitor            22080 0.11          33.9                      20
    2 content_6eeb07cca243 client_7f2253d7e2        0.891       high                 none              monitor            24516 0.09          26.9                      20
    3 content_60e8eba31e41 client_7f2253d7e2        0.890       high low_ctr_for_position review_title_snippet            27173 0.04          37.3                      20
    4 content_12c48bcc972c client_7f2253d7e2        0.889       high low_ctr_for_position review_title_snippet            10171 0.03          28.0                      20
    5 content_4718250efb96 client_7f2253d7e2        0.888       high                 n

### Recommendations (the action playbook)

- **High confidence + `low_ctr_for_position`** → `review_title_snippet` first. This is the group where both
  the transparent rule and the model agree, and both signal checks (CTR-vs-position CONFIRMED) support it.
- **High model score, but `baseline_reason_code = none`** → the model sees a broader pattern (age, low
  impressions relative to position, a weak keyword-market context) that the single-signal baseline rule
  misses. Route these to `monitor` with a note, not straight to a title rewrite — the reason is less specific.
- **Client-dominated clusters in the top of the queue** → a one-time site-level check (template, tracking,
  bulk update date) before assigning many individual page reviews, exactly as `w04`'s top-10 review found.
  **This is not hypothetical here: the top 50 rows of the deployed queue above are all one client.** Handing
  a reviewer this queue as-is would mean reviewing one client's pages fifty times before seeing a second
  client. A practical fix is to cap how many consecutive slots one client can hold (e.g. round-robin the top
  N per client, or interleave) before the queue reaches a reviewer — a policy choice for whoever owns the
  review capacity, not a modeling change.
- **Low confidence** → leave as `monitor`; do not spend limited review time here.

This ranking should be spot-checked against review capacity: if a team can review 50 pages a cycle, only the
top 50 by `model_score` should enter the queue, in that order.

## 10. Limitations, in careful words

- **The label is a same-window proxy**, not an observed future outcome (see `w03`'s contract). Everything
  here is *directional*, not causal: "associated with decline in this dataset", never "this change would fix
  the page".
- **Validated in-sample, on one static dataset.** The 20-split repetition addresses which-clients-land-where
  noise, but every split still trains and tests on the same 30,000-row snapshot — a genuinely new month of
  data, as built in `w03_data_contract.ipynb` on the warehouse release, is the stronger test the capstone
  reproducibility section points reviewers toward.
- **32 clients is a small, unbalanced panel** for a client-holdout design — some test folds saw as few as a
  handful of clients, which is exactly why Precision@50 carries a wide error bar here.
- **The gain over baseline is real but modest** (AUC ≈0.64 vs ≈0.57): worth using to order a review queue,
  not worth treating as a confident diagnosis of any single page.
- **No causal or algorithm claims.** This ranks candidates for human review; it does not claim a title fix
  will reverse a decline, and it says nothing about Google's ranking algorithm.